In [1]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 0 ns (started: 2026-10-02 21:27:40 +05:30)


In [2]:
import tensorflow as tf
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import random

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import layers, models, regularizers
from sklearn.model_selection import StratifiedKFold

from sklearn.utils.class_weight import compute_class_weight

time: 23.7 s (started: 2026-10-02 21:27:41 +05:30)


In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
time: 0 ns (started: 2026-10-02 21:28:04 +05:30)


In [4]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" 

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

MODELS_DIR = PROJECT_ROOT / "models" / "phase1"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists(), MODELS_DIR.exists())

True True True True
time: 0 ns (started: 2026-10-02 21:28:04 +05:30)


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"
SEED = 42

time: 0 ns (started: 2026-10-02 21:28:05 +05:30)


In [6]:
CLASSES = ["NORMAL", "PNEUMONIA"]

time: 0 ns (started: 2026-10-02 21:28:05 +05:30)


In [7]:
bad_files = (
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpeg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.png"))
)

deleted = 0

for f in bad_files:
    try:
        f.unlink()
        deleted += 1
    except Exception as e:
        print(f"Error deleting {f}: {e}")

print(f"Deleted {deleted} augmented files")

Deleted 0 augmented files
time: 16 ms (started: 2026-10-02 21:28:05 +05:30)


In [8]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

time: 0 ns (started: 2026-10-02 21:28:05 +05:30)


In [9]:
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

time: 0 ns (started: 2026-10-02 21:28:05 +05:30)


In [10]:
def make_balanced_fold_gen(paths, labels, batch_size=32, datagen=None):
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]

    while True:
        batch_paths, batch_labels = [], []

        for _ in range(batch_size // 2):
            p = np.random.choice(pos_idx)
            n = np.random.choice(neg_idx)

            batch_paths += [paths[p], paths[n]]
            batch_labels += [1, 0]

        combined = list(zip(batch_paths, batch_labels))
        random.shuffle(combined)
        batch_paths, batch_labels = zip(*combined)

        images = []
        for path in batch_paths:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (224, 224))
            img = img / 255.0
        
            img = np.expand_dims(img, axis=-1)  
        
            if datagen is not None:
                img = datagen.random_transform(img)  
        
            images.append(img)

        yield np.array(images), np.array(batch_labels)

time: 16 ms (started: 2026-10-02 21:28:05 +05:30)


In [11]:
label_to_classname = {0: "NORMAL", 1: "PNEUMONIA"}

time: 0 ns (started: 2026-10-02 21:28:05 +05:30)


In [12]:
def make_val_generator(val_paths, val_labels):

    val_df = pd.DataFrame({
        "filename": val_paths,
        "class": [label_to_classname[l] for l in val_labels]
    })

    return val_test_datagen.flow_from_dataframe(
        val_df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        color_mode=COLOR_MODE,
        shuffle=False
    )

time: 0 ns (started: 2026-10-02 21:28:05 +05:30)


In [13]:
def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        layers.Conv2D(32, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.1),

        layers.Conv2D(64, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])

    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )

    return m

time: 0 ns (started: 2026-10-02 21:28:05 +05:30)


In [14]:
VALID_EXTS = {".jpeg", ".jpg", ".png"}
 
all_paths = []
all_labels = []

temp_gen = train_datagen.flow_from_directory( #dummy generator
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode="binary",
    shuffle=False
)

class_map = temp_gen.class_indices  
 
for class_name, label in class_map.items():
    class_dir = TRAIN_DIR / class_name
 
    for img_name in os.listdir(class_dir):
        if Path(img_name).suffix.lower() not in VALID_EXTS:
            continue
        all_paths.append(str(class_dir / img_name))
        all_labels.append(label)
 
all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
 
print("Total samples:", len(all_paths))
print("Class distribution:", np.bincount(all_labels))

Found 4693 images belonging to 2 classes.
Total samples: 4693
Class distribution: [1206 3487]
time: 1.19 s (started: 2026-10-02 21:28:05 +05:30)


In [15]:
skf = StratifiedKFold(
    n_splits=3,  
    shuffle=True,
    random_state=42
)

time: 0 ns (started: 2026-10-02 21:28:06 +05:30)


In [16]:
for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels), 1):

    print(f"\n------------ Fold {fold} ------------")

    train_paths_fold = all_paths[train_idx]
    val_paths_fold   = all_paths[val_idx]

    train_labels_fold = all_labels[train_idx]
    val_labels_fold   = all_labels[val_idx]

    train_gen = make_balanced_fold_gen(
        train_paths_fold,
        train_labels_fold,
        batch_size=BATCH_SIZE,
        datagen=train_datagen
    )

    val_gen = make_val_generator(val_paths_fold, val_labels_fold)

    model = build_model()

    steps_per_epoch = len(train_paths_fold) // BATCH_SIZE

    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,
            patience=2,
            min_lr=1e-6
        ),
        ModelCheckpoint(
            filepath=str(MODELS_DIR / f"best_model_fold_{fold}.h5"),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

    history = model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_gen,
        epochs=15,
        callbacks=callbacks
    )



------------ Fold 1 ------------
Found 1565 validated image filenames belonging to 2 classes.
Epoch 1/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6274 - loss: 2.6070 - precision: 0.6259 - recall: 0.6321
Epoch 1: val_accuracy improved from None to 0.74313, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_fold_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 444s 4s/step - accuracy: 0.7384 - loss: 1.1069 - precision: 0.7348 - recall: 0.7461 - val_accuracy: 0.7431 - val_loss: 0.5669 - val_precision: 0.7431 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8839 - loss: 0.2946 - precision: 0.8851 - recall: 0.8827
Epoch 2: val_accuracy did not improve from 0.74313
97/97 ━━━━━━━━━━━━━━━━━━━━ 429s 4s/step - accuracy: 0.8937 - loss: 0.2797 - precision: 0.8988 - recall: 0.8872 - val_accuracy: 0.7431 - val_loss: 0.5419 - val_precision: 0.7431 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9126 - loss: 0.2405 - precision: 0.9077 - recall: 0.9191
Epoch 3: val_accuracy did not improve from 0.74313
97/97 ━━━━━━━━━━━━━━━━━━━━ 427s 4s/step - accuracy: 0.9114 - loss: 0.2489 - precision: 0.9106 - recall: 0.9124 - val_accuracy: 0.7431 - val_loss: 0.4736 - val_precision: 0.7431 - val_recall: 1.0000 - learni

97/97 ━━━━━━━━━━━━━━━━━━━━ 414s 4s/step - accuracy: 0.9204 - loss: 0.2312 - precision: 0.9251 - recall: 0.9149 - val_accuracy: 0.7853 - val_loss: 0.3473 - val_precision: 0.7762 - val_recall: 0.9991 - learning_rate: 1.0000e-04
Epoch 5/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9170 - loss: 0.2200 - precision: 0.9245 - recall: 0.9082
Epoch 5: val_accuracy improved from 0.78530 to 0.86454, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_fold_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 391s 4s/step - accuracy: 0.9224 - loss: 0.2115 - precision: 0.9243 - recall: 0.9201 - val_accuracy: 0.8645 - val_loss: 0.2687 - val_precision: 0.8484 - val_recall: 0.9957 - learning_rate: 1.0000e-04
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9242 - loss: 0.2027 - precision: 0.9272 - recall: 0.9209
Epoch 6: val_accuracy improved from 0.86454 to 0.86709, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_fold_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 354s 4s/step - accuracy: 0.9291 - loss: 0.1950 - precision: 0.9325 - recall: 0.9253 - val_accuracy: 0.8671 - val_loss: 0.2937 - val_precision: 0.8529 - val_recall: 0.9923 - learning_rate: 1.0000e-04
Epoch 7/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9336 - loss: 0.1946 - precision: 0.9385 - recall: 0.9281
Epoch 7: val_accuracy improved from 0.86709 to 0.94249, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_fold_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 372s 4s/step - accuracy: 0.9256 - loss: 0.2067 - precision: 0.9297 - recall: 0.9207 - val_accuracy: 0.9425 - val_loss: 0.1430 - val_precision: 0.9882 - val_recall: 0.9338 - learning_rate: 1.0000e-04
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9182 - loss: 0.2385 - precision: 0.9257 - recall: 0.9096
Epoch 8: val_accuracy did not improve from 0.94249
97/97 ━━━━━━━━━━━━━━━━━━━━ 388s 4s/step - accuracy: 0.9117 - loss: 0.2431 - precision: 0.9198 - recall: 0.9021 - val_accuracy: 0.9348 - val_loss: 0.1604 - val_precision: 0.9880 - val_recall: 0.9235 - learning_rate: 1.0000e-04
Epoch 9/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9262 - loss: 0.2109 - precision: 0.9380 - recall: 0.9126
Epoch 9: val_accuracy did not improve from 0.94249
97/97 ━━━━━━━━━━━━━━━━━━━━ 355s 4s/step - accuracy: 0.9272 - loss: 0.2048 - precision: 0.9414 - recall: 0.9111 - val_accuracy: 0.9042 - val_loss: 0.2341 - val_precision: 0.9894 - val_recall: 0.8805 - learni

... (5 outputs left out)

97/97 ━━━━━━━━━━━━━━━━━━━━ 246s 3s/step - accuracy: 0.8634 - loss: 0.3290 - precision_2: 0.8801 - recall_2: 0.8415 - val_accuracy: 0.7436 - val_loss: 0.4738 - val_precision_2: 0.7434 - val_recall_2: 1.0000 - learning_rate: 1.0000e-04
Epoch 5/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8776 - loss: 0.3036 - precision_2: 0.9115 - recall_2: 0.8364
Epoch 5: val_accuracy improved from 0.74361 to 0.85806, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_fold_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 197s 2s/step - accuracy: 0.8740 - loss: 0.2991 - precision_2: 0.9062 - recall_2: 0.8344 - val_accuracy: 0.8581 - val_loss: 0.2856 - val_precision_2: 0.8426 - val_recall_2: 0.9948 - learning_rate: 1.0000e-04
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8880 - loss: 0.2988 - precision_2: 0.9268 - recall_2: 0.8428
Epoch 6: val_accuracy improved from 0.85806 to 0.92327, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_fold_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.8892 - loss: 0.2913 - precision_2: 0.9242 - recall_2: 0.8479 - val_accuracy: 0.9233 - val_loss: 0.1953 - val_precision_2: 0.9155 - val_recall_2: 0.9880 - learning_rate: 1.0000e-04
Epoch 7/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8813 - loss: 0.2945 - precision_2: 0.9346 - recall_2: 0.8200
Epoch 7: val_accuracy improved from 0.92327 to 0.92583, saving model to X:\AI-Projects\Chest-Xray-Pneumonia-Detection\models\phase1\best_model_fold_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 216s 2s/step - accuracy: 0.8808 - loss: 0.2904 - precision_2: 0.9371 - recall_2: 0.8164 - val_accuracy: 0.9258 - val_loss: 0.1740 - val_precision_2: 0.9888 - val_recall_2: 0.9105 - learning_rate: 1.0000e-04
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8832 - loss: 0.2910 - precision_2: 0.9301 - recall_2: 0.8290
Epoch 8: val_accuracy did not improve from 0.92583
97/97 ━━━━━━━━━━━━━━━━━━━━ 129s 1s/step - accuracy: 0.8860 - loss: 0.2812 - precision_2: 0.9341 - recall_2: 0.8305 - val_accuracy: 0.8069 - val_loss: 0.3728 - val_precision_2: 0.9954 - val_recall_2: 0.7435 - learning_rate: 1.0000e-04
Epoch 9/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8771 - loss: 0.3107 - precision_2: 0.9257 - recall_2: 0.8202
Epoch 9: val_accuracy did not improve from 0.92583
97/97 ━━━━━━━━━━━━━━━━━━━━ 133s 1s/step - accuracy: 0.8876 - loss: 0.2849 - precision_2: 0.9381 - recall_2: 0.8299 - val_accuracy: 0.8932 - val_loss: 0.2170 - val_precision_2: 0.9931

In [17]:
print("end")

end
time: 16 ms (started: 2026-10-03 00:33:47 +05:30)


In [18]:
#changed callback to val_accuracy

time: 0 ns (started: 2026-10-03 00:33:47 +05:30)
